# 01 - WoSIS Soil Profile Exploration

This notebook explores the World Soil Information Service (WoSIS) dataset for foundation model pre-training.

## Objectives
- Understand the structure and coverage of WoSIS data
- Analyze property distributions and correlations
- Assess data quality and missing values
- Identify depth patterns and aggregation strategies

## Data Source
- WoSIS: https://www.isric.org/explore/wosis
- ~230,000 soil profiles worldwide
- Standardized soil properties at multiple depths

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project imports
from data.wosis_downloader import WoSISDownloader, WoSISConfig

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

DATA_DIR = Path('../../data/raw/wosis')
RESULTS_DIR = Path('../../results/data_analysis')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load WoSIS Data

Download the WoSIS snapshot if not already cached.

In [ ]:
# Initialize downloader
config = WoSISConfig(local_cache=str(DATA_DIR))
downloader = WoSISDownloader(config)

# Check for existing data
snapshot_path = DATA_DIR / 'wosis_snapshot.parquet'

if snapshot_path.exists():
    print(f"Loading cached data from {snapshot_path}")
    df = pd.read_parquet(snapshot_path)
else:
    print("Downloading WoSIS snapshot...")
    # Uncomment to download:
    # raw_path = downloader.download_snapshot()
    # df = downloader.process_snapshot(raw_path)
    # df.to_parquet(snapshot_path)
    print("Note: Run download separately or use sample data")
    df = pd.DataFrame()  # Placeholder

In [ ]:
# Basic statistics
print(f"Number of profiles: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

## 2. Geographic Distribution

In [ ]:
# Plot global distribution
fig, ax = plt.subplots(figsize=(14, 7))

if not df.empty and 'latitude' in df.columns:
    ax.scatter(
        df['longitude'], df['latitude'],
        s=1, alpha=0.3, c='blue'
    )
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('WoSIS Profile Locations')
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
else:
    ax.text(0.5, 0.5, 'No data loaded', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'wosis_locations.png', dpi=150)
plt.show()

In [ ]:
# Regional density analysis
if not df.empty and 'latitude' in df.columns:
    # Create latitude/longitude bins
    df['lat_bin'] = pd.cut(df['latitude'], bins=18, labels=False)
    df['lon_bin'] = pd.cut(df['longitude'], bins=36, labels=False)
    
    density = df.groupby(['lat_bin', 'lon_bin']).size().unstack(fill_value=0)
    
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.heatmap(density, cmap='YlOrRd', ax=ax)
    ax.set_title('Profile Density by Region')
    ax.set_xlabel('Longitude Bin')
    ax.set_ylabel('Latitude Bin')
    plt.tight_layout()
    plt.show()

## 3. Property Distributions

In [ ]:
# Key soil properties to analyze
PROPERTIES = ['SOC', 'pH', 'clay', 'sand', 'silt', 'N', 'CEC']

if not df.empty:
    available_props = [p for p in PROPERTIES if p in df.columns]
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i, prop in enumerate(available_props):
        if i < len(axes):
            ax = axes[i]
            df[prop].dropna().hist(bins=50, ax=ax, alpha=0.7)
            ax.set_title(f'{prop} Distribution')
            ax.set_xlabel(prop)
            ax.set_ylabel('Count')
    
    # Hide unused axes
    for i in range(len(available_props), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'wosis_property_distributions.png', dpi=150)
    plt.show()

In [ ]:
# Property statistics
if not df.empty and len(available_props) > 0:
    stats = df[available_props].describe()
    print("Property Statistics:")
    display(stats)

## 4. Property Correlations

In [ ]:
# Correlation matrix
if not df.empty and len(available_props) > 1:
    corr = df[available_props].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f',
        cmap='RdBu_r', center=0, ax=ax,
        vmin=-1, vmax=1
    )
    ax.set_title('Soil Property Correlations')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'wosis_correlations.png', dpi=150)
    plt.show()

## 5. Depth Pattern Analysis

Analyze how properties vary with depth - important for depth aggregation strategy.

In [ ]:
# If depth information is available
# This would analyze how SOC, pH, etc. change with depth

# Placeholder for depth analysis
print("Depth analysis:")
print("- Typical pattern: SOC decreases with depth")
print("- pH may increase or decrease depending on soil type")
print("- Clay often increases with depth (argillic horizon)")
print("\nRecommended aggregation: Weighted average by inverse depth")

## 6. Data Quality Assessment

In [ ]:
# Missing value analysis
if not df.empty:
    missing = df[available_props].isnull().sum() / len(df) * 100
    
    fig, ax = plt.subplots(figsize=(10, 5))
    missing.sort_values(ascending=True).plot(kind='barh', ax=ax)
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Property')
    plt.tight_layout()
    plt.show()
    
    # Profiles with complete data
    complete = df[available_props].dropna()
    print(f"\nProfiles with all properties: {len(complete):,} ({100*len(complete)/len(df):.1f}%)")

In [ ]:
# Outlier detection
if not df.empty:
    outliers = {}
    for prop in available_props:
        q1 = df[prop].quantile(0.25)
        q3 = df[prop].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        n_outliers = ((df[prop] < lower) | (df[prop] > upper)).sum()
        outliers[prop] = n_outliers
    
    print("Outliers by property (IQR method):")
    for prop, count in outliers.items():
        print(f"  {prop}: {count:,} ({100*count/len(df):.1f}%)")

## 7. Temporal Coverage

In [ ]:
# Sampling date distribution
if not df.empty and 'date' in df.columns:
    df['year'] = pd.to_datetime(df['date']).dt.year
    
    fig, ax = plt.subplots(figsize=(12, 5))
    df['year'].value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of Profiles')
    ax.set_title('WoSIS Profiles by Sampling Year')
    plt.tight_layout()
    plt.show()
    
    print(f"\nYear range: {df['year'].min()} - {df['year'].max()}")

## 8. Summary and Recommendations

In [ ]:
# Generate summary
summary = {
    'n_profiles': len(df) if not df.empty else 0,
    'n_properties': len(available_props) if not df.empty else 0,
    'geographic_coverage': 'Global' if not df.empty else 'N/A',
    'recommendations': {
        'depth_aggregation': 'Weighted average by inverse depth',
        'missing_imputation': 'Use SoilGrids predictions as gap-fill',
        'outlier_handling': 'Winsorize at 1st/99th percentiles',
        'coordinate_precision': 'Round to 4 decimal places'
    }
}

print("=" * 50)
print("WOSIS EXPLORATION SUMMARY")
print("=" * 50)
print(f"Total profiles: {summary['n_profiles']:,}")
print(f"Properties: {summary['n_properties']}")
print(f"Coverage: {summary['geographic_coverage']}")
print("\nRecommendations:")
for key, value in summary['recommendations'].items():
    print(f"  - {key}: {value}")

In [ ]:
# Save summary
import json

with open(RESULTS_DIR / 'wosis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved to {RESULTS_DIR / 'wosis_summary.json'}")